## Raw data to csv file

This notebook reads data from raw folder, sensors used by individuals (mostly surfers) and save it in csv file.

In [ ]:
import sys
sys.path.append('/Users/ignasivalles/Oceanography/IEO/projects/pom-cost.github.io/src')
from data_functions import *
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
hv.extension('bokeh')

#-54.415977, -71.004693

# === Load Historical Data ===
folder_path = "/Users/ignasivalles/Library/CloudStorage/GoogleDrive-ignasi.valles@gmail.com/La meva unitat/POM/data/raw/"
historical_data = "/Users/ignasivalles/Library/CloudStorage/GoogleDrive-ignasi.valles@gmail.com/La meva unitat/POM/data/muestreoCubo1970_78.xlsx"

# Load and preprocess historical data
df = pd.read_excel(historical_data).rename(columns={
    'año': 'year', 'mes': 'month', 'dia': 'day', 'temperatura agua': 'temperatura'
})
df['Date'] = pd.to_datetime(df[['year', 'month', 'day']])

# === 1. Climatology Calculation ===
climatology = df.groupby('month')['temperatura'].median().reset_index()
climatology['fractional_time'] = climatology['month'] - 1  # 0 for Jan, 11 for Dec

# === 2. Load Observational Data ===
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

all_data = pd.concat([get_data_from_temp_sensors(file) for file in csv_files], ignore_index=True)

# Ensure date format and extract fractional time
all_data['Date'] = pd.to_datetime(all_data['Date'])
all_data['Temperature'] = all_data['Temperature'].round(2)

# === 3. Calculate Fractional Time ===
all_data['fractional_time'] = (
    all_data['Date'].dt.month - 1 + (all_data['Date'].dt.day - 1) / all_data['Date'].dt.days_in_month
)

# === 4. Interpolate Climatology & Calculate Anomalies ===
all_data['climatology_temp'] = np.interp(
    all_data['fractional_time'],
    climatology['fractional_time'],
    climatology['temperatura']
)

all_data['Temperature_Anomaly'] = all_data['Temperature'] - all_data['climatology_temp']

# === 5. Save the Result ===
all_data.to_csv('/Users/ignasivalles/Oceanography/IEO/projects/pom-cost.github.io/data/individual_data.csv', index=False)


In [4]:
all_data

,Date,Latitude,Longitude,Temperature,fractional_time,Team,climatology_temp,Temperature_Anomaly
0,2025-01-14 14:00:00,43.450161,-3.962886,13.48,0.419355,raw,10.941935,2.538065
1,2026-02-23 11:00:00,43.465620,-3.778257,13.24,1.785714,raw,11.314286,1.925714
2,2025-05-26 18:00:00,43.458921,-3.726481,17.22,4.806452,raw,15.574194,1.645806
3,2025-08-03 09:00:00,43.450414,-3.963184,21.08,7.064516,raw,19.325806,1.754194
4,2025-10-29 08:00:00,43.463716,-3.781528,17.67,9.903226,raw,13.922581,3.747419
...,...,...,...,...,...,...,...,...
166,2025-04-08 12:00:00,43.464409,-3.780657,16.55,3.233333,raw,12.765000,3.785000
167,2025-02-20 10:00:00,43.451134,-3.821608,14.00,1.678571,raw,11.271429,2.728571
168,2024-12-06 13:00:00,43.389948,-4.384536,15.56,11.161290,raw,12.000000,3.560000
169,2024-12-15 13:00:00,43.476000,-3.694100,13.41,11.451613,raw,12.000000,1.410000


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load Historical Data ===
folder_path = "/Users/ignasivalles/Library/CloudStorage/GoogleDrive-ignasi.valles@gmail.com/La meva unitat/POM/data/raw/"
historical_data = "/Users/ignasivalles/Library/CloudStorage/GoogleDrive-ignasi.valles@gmail.com/La meva unitat/POM/data/muestreoCubo1970_78.xlsx"

df = pd.read_excel(historical_data, header=0)
df = df.rename(columns={'año': 'year', 'mes': 'month', 'dia': 'day', 'temperatura agua': 'temperatura'})

# Create a Date column
df['Date'] = pd.to_datetime(df[['year', 'month', 'day']])
df['Date'] = df['Date'].astype('datetime64[ns]')

# === 1. Create Climatological Time Series ===
# Group by month for climatological statistics
climatology = df.groupby('month')['temperatura'].median().reset_index()
climatology = climatology.rename(columns={'temperatura': 'climatology_temp'})

# Add fractional time for each month (0 to 11)
climatology['fractional_time'] = climatology['month'] - 1  # Adjust to start from 0

# === 2. Load Observational Data ===
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

all_data = []
for file in csv_files:
    idf1 = get_data_from_temp_sensors(file)
    all_data.append(idf1)

all_data = pd.concat(all_data, ignore_index=True)
all_data['Temperature'] = all_data['Temperature'].round(2)
all_data['Date'] = pd.to_datetime(all_data['Date'])

# === 3. Calculate Fractional Time for Observational Data ===
all_data['month'] = all_data['Date'].dt.month
all_data['day'] = all_data['Date'].dt.day

# Fractional time: month - 1 + (day - 1) / days_in_month
all_data['days_in_month'] = all_data['Date'].dt.days_in_month
all_data['fractional_time'] = (all_data['month'] - 1) + (all_data['day'] - 1) / all_data['days_in_month']

# === 4. Interpolate Climatology ===
# Interpolation function
interp_func = np.interp(
    all_data['fractional_time'], 
    climatology['fractional_time'], 
    climatology['climatology_temp']
)

# Add interpolated climatology to all_data
all_data['interpolated_climatology_temp'] = interp_func

# === 5. Calculate Temperature Anomaly ===
all_data['Temperature_Anomaly'] = all_data['Temperature'] - all_data['interpolated_climatology_temp']

# === 6. Save the Result ===
all_data.to_csv('/Users/ignasivalles/Oceanography/IEO/projects/POM/pom.github.io/individual_data.csv', index=False)


In [6]:
all_data

,Date,Latitude,Longitude,Temperature,fractional_time,Team,month,day,days_in_month,interpolated_climatology_temp,Temperature_Anomaly
0,2025-01-14 15:00:00,43.450161,-3.962886,13.48,0.419355,raw,1,14,31,10.941935,2.538065
1,2025-01-10 15:00:00,43.476076,-3.694162,13.63,0.290323,raw,1,10,31,10.929032,2.700968
2,2025-02-02 07:00:00,43.389521,-4.382791,12.85,1.035714,raw,2,2,28,11.014286,1.835714
3,2025-01-02 16:00:00,37.108780,-8.935526,16.91,0.032258,raw,1,2,31,10.903226,6.006774
4,2024-12-13 14:00:00,43.455761,-3.742693,14.73,11.387097,raw,12,13,31,12.000000,2.730000
5,2024-12-07 13:00:00,43.586608,-5.760915,14.85,11.193548,raw,12,7,31,12.000000,2.850000
6,2024-09-23 09:00:00,43.463410,-3.782722,16.80,8.733333,raw,9,23,30,16.600000,0.200000
7,2024-09-24 18:00:00,43.449935,-3.965992,19.73,8.766667,raw,9,24,30,16.525000,3.205000
8,2024-09-23 09:00:00,43.460000,-3.780000,16.80,8.733333,raw,9,23,30,16.600000,0.200000
9,2024-12-01 08:00:00,43.450000,-3.970000,15.28,11.000000,raw,12,1,31,12.000000,3.280000


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load Historical Data ===
folder_path = "/Users/ignasivalles/Library/CloudStorage/GoogleDrive-ignasi.valles@gmail.com/La meva unitat/POM/data/raw/"
historical_data = "/Users/ignasivalles/Library/CloudStorage/GoogleDrive-ignasi.valles@gmail.com/La meva unitat/POM/data/muestreoCubo1970_78.xlsx"

# Load and preprocess historical data
df = pd.read_excel(historical_data).rename(columns={
    'año': 'year', 'mes': 'month', 'dia': 'day', 'temperatura agua': 'temperatura'
})
df['Date'] = pd.to_datetime(df[['year', 'month', 'day']])

# === 1. Climatology Calculation ===
climatology = df.groupby('month')['temperatura'].median().reset_index()
climatology['fractional_time'] = climatology['month'] - 1  # 0 for Jan, 11 for Dec

# === 2. Load Observational Data ===
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
all_data = pd.concat([get_data_from_temp_sensors(file) for file in csv_files], ignore_index=True)

# Ensure date format and extract fractional time
all_data['Date'] = pd.to_datetime(all_data['Date'])
all_data['Temperature'] = all_data['Temperature'].round(2)

# === 3. Calculate Fractional Time ===
all_data['fractional_time'] = (
    all_data['Date'].dt.month - 1 + (all_data['Date'].dt.day - 1) / all_data['Date'].dt.days_in_month
)

# === 4. Interpolate Climatology & Calculate Anomalies ===
all_data['climatology_temp'] = np.interp(
    all_data['fractional_time'],
    climatology['fractional_time'],
    climatology['temperatura']
)

all_data['Temperature_Anomaly'] = all_data['Temperature'] - all_data['climatology_temp']

# === 5. Save the Result ===
all_data.to_csv('/Users/ignasivalles/Oceanography/IEO/projects/POM/pom.github.io/individual_data.csv', index=False)


In [3]:
all_data

,Date,Latitude,Longitude,Temperature,fractional_time,Team,climatology_temp,Temperature_Anomaly
0,2025-01-14 15:00:00,43.450161,-3.962886,13.48,0.419355,raw,10.941935,2.538065
1,2025-01-10 15:00:00,43.476076,-3.694162,13.63,0.290323,raw,10.929032,2.700968
2,2025-02-02 07:00:00,43.389521,-4.382791,12.85,1.035714,raw,11.014286,1.835714
3,2025-01-02 16:00:00,37.108780,-8.935526,16.91,0.032258,raw,10.903226,6.006774
4,2024-12-13 14:00:00,43.455761,-3.742693,14.73,11.387097,raw,12.000000,2.730000
5,2024-12-07 13:00:00,43.586608,-5.760915,14.85,11.193548,raw,12.000000,2.850000
6,2024-09-23 09:00:00,43.463410,-3.782722,16.80,8.733333,raw,16.600000,0.200000
7,2024-09-24 18:00:00,43.449935,-3.965992,19.73,8.766667,raw,16.525000,3.205000
8,2024-12-01 08:00:00,43.450000,-3.970000,15.28,11.000000,raw,12.000000,3.280000
9,2025-01-17 16:00:00,43.462663,-3.724438,13.41,0.516129,raw,10.951613,2.458387


np.float64(0.9205479452054794)